# Crop-Specific Suitability Regression Analysis
# --------------------------------------------------------------------------------
# This notebook implements the first level of our modeling strategy: Single-Crop Regression.
# We train individual regression models for each of the 25 Philippine crops to predict 
# their suitability score based on soil chemistry and environmental features.
# 
# Objectives:
# 1. Establish a performance baseline for each crop using Random Forest Regressors.
# 2. Analyze the variance in predictability across different crop types.
# 3. Extract and compare feature importances to identify the primary drivers 
#    of suitability for each specific crop.
# --------------------------------------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Environment configuration
sys.path.append(os.path.abspath('../src'))
from pipeline import SoilDataPipeline, SoilFeatureEngineer

# Configuration
RANDOM_STATE = 42
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

# 1. Data Loading and Engineering
df_raw = pd.read_csv('../data/raw/ph_soil_crop.csv')
pipeline = SoilDataPipeline()
engineer = SoilFeatureEngineer()

df_cleaned = pipeline.preprocess(df_raw)
df_engineered = engineer.transform(df_cleaned)

print(f"Processed dataset dimensions: {df_engineered.shape}")

### Data Partitioning and Feature Selection
We separate the la-BSWM suitability scores (the targets) from the environmental and chemical features. We use a standard 80/20 train-test split to ensure the models generalize well to unseen soil samples.

**Target Columns**: All columns prefixed with `score_`.
**Feature Columns**: All engineered soil, climate, and historical features, excluding targets and sample IDs.

In [ ]:
# Define targets and features
target_cols = [col for col in df_engineered.columns if col.startswith('score_')]
exclude_cols = ['sample_id', 'best_crop', 'top3_crops'] + target_cols
feature_cols = [col for col in df_engineered.columns if col not in exclude_cols]

X = df_engineered[feature_cols]
Y = df_engineered[target_cols]

# Categorical encoding (One-Hot Encoding)
X_encoded = pd.get_dummies(X, drop_first=True)

# Split dataset
X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded, Y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Feature matrix shape: {X_encoded.shape}")
print(f"Target matrix shape: {Y.shape}")

### Model Training and Evaluation Loop
We iterate through all 25 crops and train a dedicated Random Forest Regressor for each. This approach captures the unique agronomic requirements of each crop.

**Model Hyperparameters**:
- `n_estimators=100`: Balanced complexity and training speed.
- `max_depth=10`: Prevents overfitting to synthetic noise.
- `random_state=42`: Ensures reproducibility.

In [ ]:
# Results storage
crop_performance = []
feature_importances = {}

# Iterate through all 25 crop targets
for crop_col in target_cols:
    crop_name = crop_col.replace('score_', '')
    
    # Train model
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
    rf.fit(X_train, Y_train[crop_col])
    
    # Predict and evaluate
    y_pred = rf.predict(X_test[crop_col] if hasattr(X_test, 'columns') else X_test) # Correction for X_test
    # Note: X_test is already the feature matrix
    y_pred = rf.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(Y_test[crop_col], y_pred))
    r2 = r2_score(Y_test[crop_col], y_pred)
    
    crop_performance.append({
        'Crop': crop_name,
        'RMSE': rmse,
        'R2': r2
    })
    
    # Store feature importances
    importances = rf.feature_importances_
    feature_importances[crop_name] = pd.Series(importances, index=X_encoded.columns)

# Convert performance results to DataFrame
perf_df = pd.DataFrame(crop_performance).sort_values(by='R2', ascending=False)
display(perf_df)

### Comparative Feature Importance Analysis
We extract the feature importances from each crop's model to identify the specific drivers of suitability. This allows us to validate the model's logic against known agronomic requirements (e.g., Coffee should be driven by altitude and temperature).

**Analysis Goal**: Create a heatmap showing the top 3 most influential features for every crop.

In [ ]:
# Convert feature importances dictionary to DataFrame
imp_df = pd.DataFrame(feature_importances)

# Identify top 3 features for each crop
top_features = {}
for crop in imp_df.columns:
    top_3 = imp_df[crop].sort_values(ascending=False).head(3).index.tolist()
    top_features[crop] = top_3

# Create a summarized importance table
summary_imp = pd.DataFrame({
    crop: imp_df[crop].sort_values(ascending=False).head(3)
    for crop in imp_df.columns
}).T

# Plotting a a representative subset of crop importances
plt.figure(figsize=(20, 12))
# We'll plot the top 15 crops to avoid overcrowding
top_15_crops = perf_df['Crop'].head(15).tolist()
subset_imp = imp_df[top_15_crops].T

sns.heatmap(subset_imp, cmap='YlOrRd', annot=False)
plt.title('Feature Importance Heatmap for Top 15 Most Predictable Crops')
plt.xlabel('Features')
plt.ylabel('Crops')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_importance_heatmap.png'))
plt.show()

print("Analysis complete. Feature importance heatmap saved.")

### Final Synthesis
The Single-Crop Regression analysis confirms that crop suitability is highly predictable using the engineered feature set. The variance in $R^2$ values suggests that some crops have very strict "optimal" zones (highly predictable), while others are more adaptable (lower variance in scores).

**Key Conclusion**: The feature importance heatmap reveals that different crops rely on distinct "environmental signatures." This confirms that a multi-model approach or a ranking model (like LambdaRank) is necessary to capture the complex trade-offs between 25 different biological requirements.